In [2]:
import csv
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun

load_dotenv()

with open("../data/candidatos/candidatos.csv", encoding="utf-8-sig") as f:
    candidatos = list(csv.DictReader(f))

nome_para_id = {c["nome_urna"]: c["id_candidato"] for c in candidatos}

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
)

vectorstore = Chroma(
    persist_directory="../data/vectorstores/chroma_candidatos",
    embedding_function=embeddings,
)

busca_duckduckgo = DuckDuckGoSearchRun()


@tool
def buscar_propostas_candidato(candidato: str, pergunta: str) -> str:
    """Busca trechos das propostas de governo de um candidato à presidência do Brasil em 2026.

    Use esta ferramenta sempre que o usuário perguntar sobre propostas, planos de governo
    ou posições de um candidato específico em algum tema (educação, saúde, economia, etc).

    Args:
        candidato: o nome comum do candidato, exatamente como é conhecido (ex: "Lula", "Renan Santos").
        pergunta: a pergunta ou tema a buscar dentro do documento de propostas desse candidato.
    """
    id_candidato = nome_para_id.get(candidato)

    if id_candidato is None:
        candidatos_disponiveis = ", ".join(nome_para_id.keys())
        return f"Candidato '{candidato}' não encontrado. Candidatos disponíveis: {candidatos_disponiveis}"

    retriever = vectorstore.as_retriever(
        search_kwargs={"k": 4, "filter": {"candidato": id_candidato}}
    )
    resultados = retriever.invoke(pergunta)

    if not resultados:
        return f"Nenhum trecho encontrado sobre '{pergunta}' nas propostas de {candidato}."

    trechos_formatados = []
    for r in resultados:
        pagina = r.metadata["page"] + 1
        nome = r.metadata["nome_urna"]
        trechos_formatados.append(f"[{nome} - Página {pagina}] {r.page_content}")

    return "\n\n".join(trechos_formatados)


@tool
def buscar_informacoes_web(candidato: str, tema: str) -> str:
    """Busca na internet informações atuais sobre um candidato à presidência do Brasil em 2026.

    Use esta ferramenta apenas para perguntas que NÃO sejam sobre propostas de governo
    (isso já é coberto pela ferramenta buscar_propostas_candidato). Prefira esta ferramenta
    para: notícias recentes, biografia, trajetória política, ou quando a busca nas propostas
    não encontrou nada sobre o tema perguntado.

    Args:
        candidato: nome comum do candidato, exatamente como conhecido (ex: "Lula", "Renan Santos").
        tema: o que buscar sobre esse candidato (ex: "biografia", "últimas notícias", "trajetória política").
    """
    if candidato not in nome_para_id:
        candidatos_disponiveis = ", ".join(nome_para_id.keys())
        return f"'{candidato}' não é um dos candidatos à presidência cobertos por este projeto. Candidatos disponíveis: {candidatos_disponiveis}"

    query = f"{candidato} candidato presidente Brasil eleições 2026 {tema}"
    return busca_duckduckgo.invoke(query)


ferramentas = [buscar_propostas_candidato, buscar_informacoes_web]

print("Setup pronto.")

C:\Users\duda_\AppData\Local\Temp\ipykernel_11816\1589318746.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


Setup pronto.


2. vincular as tools ao LLM
O que essa célula faz: .bind_tools(ferramentas) não executa nada ainda — só "apresenta" as duas tools pro LLM, junto com as docstrings que você escreveu. Quando você manda uma pergunta pra esse llm_com_ferramentas, ele não responde a pergunta diretamente: ele decide (baseado no que leu nas docstrings) se quer chamar alguma tool, e devolve essa decisão estruturada em resposta.tool_calls — exatamente como você já viu acontecer no Nexus, só que agora com duas tools competindo em vez de uma.

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-3-flash-preview")

llm_com_ferramentas = llm.bind_tools(ferramentas)

resposta = llm_com_ferramentas.invoke("Quais são as propostas do Renan Santos para educação?")

print(resposta.tool_calls)

[{'name': 'buscar_propostas_candidato', 'args': {'candidato': 'Renan Santos', 'pergunta': 'educação'}, 'id': 'call_7296087', 'type': 'tool_call'}]


3. montando o grafo

In [ ]:
from langgraph.graph import StateGraph, END, MessagesState
from langgraph.prebuilt import ToolNode

def no_llm(state: MessagesState): #nó LLM(decide) - recebe o estado (a conversa ate agora)
    resposta = llm_com_ferramentas.invoke(state["messages"]) #manda para o llm_com_ferramentas e devolve a resposta acrescentada ao estado
    return {"messages": [resposta]}

def deve_continuar(state: MessagesState): #aresta condicional (chamou uma tool?)
    ultima_mensagem = state["messages"][-1] #olha a última mensagem
    if ultima_mensagem.tool_calls: #se ela tem tool_calls, manda pro nó "tools", se não manda pro END
        return "tools"
    return END

workflow = StateGraph(MessagesState)
workflow.add_node("agente", no_llm)
workflow.add_node("tools", ToolNode(ferramentas)) # "Nó tools (executa)" pronto do LangGraph. Ele ja sabe pegar o tool_calls de uma mensagem e chamar a funcao certa sozinho.

workflow.set_entry_point("agente")
workflow.add_conditional_edges("agente", deve_continuar, {"tools": "tools", END: END})
workflow.add_edge("tools", "agente") #é a seta. depois de executar a tool, sempre volta pro LLM decidir de novo.

grafo = workflow.compile()

print("Grafo montado.")

Grafo montado.


4. testando o agente

In [6]:
resultado = grafo.invoke({
    "messages": [("user", "Quais são as propostas do Renan Santos para educação?")]
})

for mensagem in resultado["messages"]:
    mensagem.pretty_print()

================================ Human Message =================================

Quais são as propostas do Renan Santos para educação?
================================== Ai Message ==================================

[]
Tool Calls:
  buscar_propostas_candidato (call_10591454)
 Call ID: call_10591454
  Args:
    candidato: Renan Santos
    pergunta: quais são as propostas para educação?
================================= Tool Message =================================
Name: buscar_propostas_candidato

[Renan Santos - Página 31] sociedade. Em um país de tradição universitária incipiente como o Brasil, que ainda possui um perfil marca-
damente elitista, a maior parte da sociedade possui somente esta escolaridade em seu limite. Portanto, nos 
concentraremos em pensar soluções frontais e exequíveis para estes dois problemas.
PROPOSTAS E SOLUÇÕES DA MISSÃO
No Brasil, o melhor exemplo de elevação da educação básica encontra-se no Ceará e é, portanto, a este 
modelo que nos remetemos, pensando 